# 🔒 IEEE-CIS Fraud Detection — Advanced Data Preprocessing Pipeline

> **Goal:** Load the raw transaction and identity CSVs, apply a rigorous object-oriented
> preprocessing pipeline, and export a clean, analysis-ready `data/processed_train.csv`.

### Pipeline Overview

| Step | Transformer | Purpose |
|------|------------|--------|
| 1 | `reduce_mem_usage` | Downcast dtypes — shrink RAM by ~50 % |
| 2 | `DataMerger` | Left-join Transaction + Identity; fix `id-XX` → `id_XX` |
| 3 | `TimeFeatureExtractor` | Extract hour-of-day & day-of-week from `TransactionDT` |
| 4 | `DropHighNulls` | Remove columns with > 85 % missing values |
| 5 | `FrequencyEncoder` | Replace categories with their relative frequency |
| 6 | Final cast | Ensure all remaining columns are numeric |
| 7 | Save | Write `data/processed_train.csv` |

---
Each transformer below is built as a **scikit-learn custom estimator** (`BaseEstimator` + `TransformerMixin`),
so the full pipeline is reusable, stateful, and leak-free when applied to the test set.

## ⚙️ Step 0 — Imports & Path Configuration

We import the standard scientific Python stack plus scikit-learn's base classes that allow us
to write custom, pipeline-compatible transformers.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
# Works whether the notebook is opened from /notebooks/ or the project root.
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
OUTPUT_CSV = os.path.join(DATA_DIR, 'processed_train.csv')

print('Project root :', BASE_DIR)
print('Data dir     :', DATA_DIR)

## 🗜️ Step 1 — Memory Optimisation (`reduce_mem_usage`)

### What it does
Iterates every numeric column and downcasts it to the smallest dtype that can safely hold its
value range (e.g. `float64` → `float16`, `int64` → `int8`).

### Why it is necessary
The raw CSVs total **~1.5 GB**. Pandas defaults to 64-bit types for everything, often wasting
4× the RAM actually required. Downcasting typically achieves a **50–70 % memory reduction**,
making local experimentation feasible on a standard laptop without kernel crashes.

In [ ]:
def reduce_mem_usage(df, verbose=True):
    """
    Automatically downcast numeric columns to the minimum safe dtype.
    Returns the modified DataFrame and prints the memory reduction.
    """
    numerics  = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype
        if col_type.name in numerics:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                for t in [np.int8, np.int16, np.int32, np.int64]:
                    if np.iinfo(t).min < c_min and c_max < np.iinfo(t).max:
                        df[col] = df[col].astype(t)
                        break
            else:
                for t in [np.float16, np.float32, np.float64]:
                    if np.finfo(t).min < c_min and c_max < np.finfo(t).max:
                        df[col] = df[col].astype(t)
                        break

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f'  RAM: {start_mem:.1f} MB  →  {end_mem:.1f} MB  '
              f'({100*(start_mem - end_mem)/start_mem:.1f}% reduction)')
    return df

## 🔀 Step 2 — Data Merger (`DataMerger`)

### What it does
Performs a **left join** of the Transaction table onto the Identity table using `TransactionID`
as the join key. It also standardises identity column names by replacing dashes with underscores
(`id-01` → `id_01`).

### Why it is necessary
Fraud footprints rely heavily on device and network metadata stored only in the identity table.
Not every transaction has associated identity data — a left join preserves all transactions.
The dash/underscore mismatch between train and test identity columns would silently break
model inference at scoring time if left unfixed.

In [ ]:
class DataMerger(BaseEstimator, TransformerMixin):
    """
    Left-join Transaction DataFrame with Identity DataFrame on 'TransactionID'.
    Also standardises identity column names (dash → underscore).
    """
    def __init__(self, identity_df):
        self.identity_df = identity_df

    def fit(self, X, y=None):
        return self  # stateless — nothing to learn

    def transform(self, X):
        print('  [DataMerger] Standardising column names and merging ...')
        id_df = self.identity_df.copy()
        # Fix the dash/underscore discrepancy between train and test identity files
        id_df.columns = [c.replace('-', '_') for c in id_df.columns]
        merged = X.merge(id_df, on='TransactionID', how='left')
        print(f'  Shape after merge: {merged.shape}')
        return merged

## ⏰ Step 3 — Temporal Feature Engineering (`TimeFeatureExtractor`)

### What it does
Derives **`Transaction_Hour`** (0–23) and **`Transaction_Day`** (0–6, Mon–Sun) from the raw
`TransactionDT` column, which is a timedelta in seconds from an unknown reference datetime.

### Why it is necessary
Raw seconds form a monotonically increasing counter that a tree model cannot easily translate
into cyclical patterns. Deriving hour-of-day and day-of-week directly exposes temporal fraud
signatures — e.g. card-testing spikes at 3 AM, or weekend automated fraud waves — with
near-zero computational cost and no additional data required.

In [ ]:
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Extract cyclical temporal features from the TransactionDT column.
    TransactionDT is assumed to be in seconds from a fixed reference datetime.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print('  [TimeFeatureExtractor] Engineering temporal features ...')
        X = X.copy()
        X['Transaction_Hour'] = (X['TransactionDT'] // 3600) % 24   # 0–23
        X['Transaction_Day']  = (X['TransactionDT'] // 86400) % 7   # 0–6
        print(f'  Added: Transaction_Hour, Transaction_Day')
        return X

## 🗑️ Step 4 — High-Null Column Pruning (`DropHighNulls`)

### What it does
During **`fit()`**, computes the fraction of missing values per column and records those
exceeding `threshold`. During **`transform()`**, drops those columns from any DataFrame
passed in (train or test).

### Why it is necessary
Many of the `V`-series anonymised features have > 90 % missing values. Imputing a column
that is 95 % NaN forces the model to learn an imputed constant instead of a real signal,
injecting significant noise. It also inflates dimensionality, worsening the
*curse of dimensionality* for distance-based or regularised models.

> **Threshold choice:** `0.85` — drop any column missing more than 85 % of its values.

In [ ]:
class DropHighNulls(BaseEstimator, TransformerMixin):
    """
    Fit: identify columns whose null fraction exceeds `threshold`.
    Transform: drop those columns.
    """
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        null_frac = X.isnull().mean()
        self.cols_to_drop_ = null_frac[null_frac > self.threshold].index.tolist()
        print(f'  [DropHighNulls] {len(self.cols_to_drop_)} columns flagged for removal '
              f'(>{self.threshold*100:.0f}% missing)')
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors='ignore')

## 🔡 Step 5 — Frequency Encoding (`FrequencyEncoder`)

### What it does
Replaces each category in an object/categorical column with its **normalised frequency** —
i.e. the fraction of rows containing that value — as learned from the *training* set.

### Why it is necessary
| Encoding | Problem |
|----------|---------|
| Label Encoding | Arbitrary integers mislead ordinal-sensitive algorithms |
| One-Hot Encoding | Hundreds of sparse binary columns — memory explosion |
| **Frequency Encoding** | Single meaningful numeric per category — rare fraud fingerprints get small values tree models can split on precisely |

Frequencies are fit on the training set and applied unchanged to the test set to **prevent
target leakage**.

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """
    Fit: compute normalised frequency per category for each categorical column.
    Transform: replace raw categories with their learned frequency.
    Unseen categories at inference time are mapped to 0.
    """
    def __init__(self, cat_cols=None):
        self.cat_cols  = cat_cols
        self.freq_maps_ = {}

    def fit(self, X, y=None):
        cols = self.cat_cols or X.select_dtypes(include=['object', 'category']).columns.tolist()
        for col in cols:
            if col in X.columns:
                self.freq_maps_[col] = (
                    X[col].value_counts(normalize=True, dropna=False).to_dict()
                )
        print(f'  [FrequencyEncoder] Fitted on {len(self.freq_maps_)} categorical columns')
        return self

    def transform(self, X):
        print(f'  [FrequencyEncoder] Encoding {len(self.freq_maps_)} columns ...')
        X = X.copy()
        for col, fmap in self.freq_maps_.items():
            if col in X.columns:
                X[col] = X[col].map(fmap).fillna(0)  # 0 for unseen categories
        return X

## 🚀 Step 6 — Load Data & Execute the Full Pipeline

Now that all transformers are defined we:
1. **Read** both raw CSVs.
2. **Memory-optimise** them immediately (before the pipeline, to keep peak RAM low).
3. **Assemble** the scikit-learn `Pipeline`.
4. **Execute** `fit_transform()` on the training transaction data.
5. **Cast** any residual object columns to numeric (safety net).
6. **Persist** to `data/processed_train.csv` — ready for EDA.

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
print('[1/5] Loading raw CSVs ...')
train_tx = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_id = pd.read_csv(os.path.join(DATA_DIR, 'train_identity.csv'))
print(f'  train_transaction : {train_tx.shape}')
print(f'  train_identity    : {train_id.shape}')

In [ ]:
# ── Memory optimisation ───────────────────────────────────────────────────────
print('[2/5] Optimising memory ...')
train_tx = reduce_mem_usage(train_tx)
train_id = reduce_mem_usage(train_id)

In [ ]:
# ── Build & run pipeline ──────────────────────────────────────────────────────
print('[3/5] Running preprocessing pipeline ...')
pipeline = Pipeline([
    ('merger',       DataMerger(identity_df=train_id)),
    ('time_extract', TimeFeatureExtractor()),
    ('drop_nulls',   DropHighNulls(threshold=0.85)),
    ('freq_encoder', FrequencyEncoder()),
])
processed = pipeline.fit_transform(train_tx)

In [ ]:
# ── Final cast & inspect ──────────────────────────────────────────────────────
print('[4/5] Final numeric cast ...')
for col in processed.select_dtypes(include=['object']).columns:
    processed[col] = pd.to_numeric(processed[col], errors='coerce')

print(f'  Final shape : {processed.shape}')
print(f'  isFraud distribution:')
print(processed['isFraud'].value_counts())
processed.head(3)

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
print(f'[5/5] Saving to {OUTPUT_CSV} ...')
processed.to_csv(OUTPUT_CSV, index=False)
print('\n✅  Preprocessing complete!  processed_train.csv is ready for EDA.')